# 01 — Data Audit

This notebook verifies the structure, data splits, eligibility rules,
target availability, and class balance of the supplied synthetic
limit-order-book dataset.

The purpose is to confirm the dataset before any exploratory analysis
or model training.

In [6]:
from pathlib import Path
import gzip
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"

excel_files = list(RAW_DIR.glob("*.xlsx"))
json_files = sorted(
    list(RAW_DIR.glob("*.json"))
    + list(RAW_DIR.glob("*.json.gz"))
)

print("Project root:", PROJECT_ROOT)
print("Excel files:", [file.name for file in excel_files])
print("JSON files:", [file.name for file in json_files])




Project root: <repository-root>
Excel files: ['AI_Order_Execution.xlsx']
JSON files: ['Single_Symbol_LOB_History.json']


In [7]:
assert len(excel_files) == 1, (
    f"Expected exactly one Excel file, found {len(excel_files)}"
)

assert len(json_files) == 1, (
    f"Expected exactly one JSON file, found {len(json_files)}"
)

EXCEL_PATH = excel_files[0]
JSON_PATH = json_files[0]

print("Excel:", EXCEL_PATH.name)
print("JSON:", JSON_PATH.name)
print("JSON file size:", round(JSON_PATH.stat().st_size / 1024**2, 2), "MB")

Excel: AI_Order_Execution.xlsx
JSON: Single_Symbol_LOB_History.json
JSON file size: 314.69 MB


In [8]:
workbook = pd.ExcelFile(EXCEL_PATH)

print("Workbook sheets:")
for sheet_name in workbook.sheet_names:
    print("-", sheet_name)

Workbook sheets:
- README
- Orders
- Data_Dictionary
- JSON_Schema
- Summary
- Quality_Checks


In [9]:
orders = pd.read_excel(
    EXCEL_PATH,
    sheet_name="Orders"
)

print("Orders shape:", orders.shape)
orders.head()

Orders shape: (10000, 72)


,order_id,trading_day,observation_timestamp,symbol,side,seconds_from_open,session_phase,order_age_s,tick_size,limit_price,...,first_fill_time_s,first_fill_quantity,first_fill_fraction,eligible_5s,target_fill_5s,eligible_30s,target_fill_30s,eligible_60s,target_fill_60s,data_split
0,ORD000001,2022-09-06,2022-09-06T09:35:45.546-04:00 (America/New_York),AAPL,buy,345.545994,opening,12.054852,0.01,143.50,...,16.024293,71.0,0.568,1,0.0,1,1.0,1,1.0,train
1,ORD000002,2022-09-06,2022-09-06T09:35:38.278-04:00 (America/New_York),AAPL,sell,338.278096,opening,3.071597,0.01,143.61,...,70.797932,1.0,1.000,1,0.0,1,0.0,1,0.0,train
2,ORD000003,2022-09-06,2022-09-06T09:36:56.721-04:00 (America/New_York),AAPL,buy,416.720830,opening,3.763735,0.01,143.51,...,128.532267,1.0,1.000,1,0.0,1,0.0,1,0.0,train
3,ORD000004,2022-09-06,2022-09-06T09:37:14.773-04:00 (America/New_York),AAPL,buy,434.773192,opening,12.956035,0.01,143.74,...,NaN,NaN,NaN,1,0.0,1,0.0,1,0.0,train
4,ORD000005,2022-09-06,2022-09-06T09:37:18.780-04:00 (America/New_York),AAPL,sell,438.779839,opening,13.545683,0.01,143.58,...,100.274516,1.0,1.000,1,0.0,1,0.0,1,0.0,train


In [10]:
print("Number of columns:", len(orders.columns))

for column in orders.columns:
    print(column)

Number of columns: 72
order_id
trading_day
observation_timestamp
symbol
side
seconds_from_open
session_phase
order_age_s
tick_size
limit_price
order_size
price_distance_ticks
quantity_filled_before_t0
queue_ahead_qty
same_price_total_qty
normalized_queue_ahead
bid_price_l1
bid_volume_l1
ask_price_l1
ask_volume_l1
bid_price_l2
bid_volume_l2
ask_price_l2
ask_volume_l2
bid_price_l3
bid_volume_l3
ask_price_l3
ask_volume_l3
bid_price_l4
bid_volume_l4
ask_price_l4
ask_volume_l4
bid_price_l5
bid_volume_l5
ask_price_l5
ask_volume_l5
mid_price
spread_ticks
relative_spread_bps
top_level_imbalance
five_level_side_imbalance
same_side_depth_5
opposite_side_depth_5
microprice
volatility_60s_bps
same_side_submissions_10s
opposite_side_submissions_10s
same_side_cancellations_10s
opposite_side_cancellations_10s
modifications_10s
aggressive_trade_volume_against_order_10s
aggressive_trade_volume_with_order_side_10s
trade_count_10s
trade_intensity_per_s
same_side_cancel_intensity_per_s
time_since_last_tra

In [11]:
split_counts = (
    orders["data_split"]
    .astype(str)
    .str.lower()
    .value_counts()
    .rename_axis("split")
    .reset_index(name="rows")
)

split_counts

,split,rows
0,train,7000
1,validation,1500
2,test,1500


In [12]:
split_count_dict = dict(
    zip(split_counts["split"], split_counts["rows"])
)

assert split_count_dict == {
    "train": 7000,
    "validation": 1500,
    "test": 1500
}

print("Chronological split counts are correct.")

Chronological split counts are correct.


In [13]:
target_audit_rows = []

for horizon in [5, 30, 60]:
    eligibility_column = f"eligible_{horizon}s"
    target_column = f"target_fill_{horizon}s"

    eligible_mask = orders[eligibility_column].eq(1)

    eligible_targets = pd.to_numeric(
        orders.loc[eligible_mask, target_column],
        errors="coerce"
    )

    ineligible_targets = pd.to_numeric(
        orders.loc[~eligible_mask, target_column],
        errors="coerce"
    )

    target_audit_rows.append({
        "horizon_seconds": horizon,
        "eligible_rows": int(eligible_mask.sum()),
        "available_targets": int(eligible_targets.notna().sum()),
        "positive_rate_pct": 100 * eligible_targets.mean(),
        "targets_present_when_ineligible": int(
            ineligible_targets.notna().sum()
        )
    })

target_audit = pd.DataFrame(target_audit_rows)
target_audit.round(2)

,horizon_seconds,eligible_rows,available_targets,positive_rate_pct,targets_present_when_ineligible
0,5,9722,9722,10.62,0
1,30,8808,8808,41.87,0
2,60,8252,8252,63.17,0


In [14]:
expected_eligible_rows = {
    5: 9722,
    30: 8808,
    60: 8252
}

expected_positive_rates = {
    5: 10.6,
    30: 41.9,
    60: 63.2
}

for row in target_audit_rows:
    horizon = row["horizon_seconds"]

    assert row["eligible_rows"] == expected_eligible_rows[horizon]
    assert row["available_targets"] == row["eligible_rows"]
    assert row["targets_present_when_ineligible"] == 0

    assert round(row["positive_rate_pct"], 1) == (
        expected_positive_rates[horizon]
    )

print("All eligibility and target checks passed.")

All eligibility and target checks passed.


In [15]:
id_audit = pd.Series({
    "total_rows": len(orders),
    "missing_order_ids": orders["order_id"].isna().sum(),
    "unique_order_ids": orders["order_id"].nunique(),
    "duplicated_order_ids": orders["order_id"].duplicated().sum()
})

id_audit

total_rows              10000
missing_order_ids           0
unique_order_ids        10000
duplicated_order_ids        0
dtype: int64

In [16]:
assert orders["order_id"].notna().all()
assert orders["order_id"].is_unique

print("Every order has a unique, non-missing order_id.")

missing_summary = (
    orders.isna()
    .sum()
    .loc[lambda values: values > 0]
    .sort_values(ascending=False)
    .rename("missing_rows")
    .to_frame()
)

missing_summary

Every order has a unique, non-missing order_id.


,missing_rows
censoring_reason,7571
first_fill_time_s,2429
first_fill_quantity,2429
first_fill_fraction,2429
target_fill_60s,1748
target_fill_30s,1192
target_fill_5s,278


In [17]:
for horizon in [5, 30, 60]:
    eligibility_column = f"eligible_{horizon}s"
    target_column = f"target_fill_{horizon}s"

    expected_missing = orders[eligibility_column].eq(0).sum()
    actual_missing = orders[target_column].isna().sum()

    assert actual_missing == expected_missing

non_fill_mask = orders["fill_event"].eq(0)
fill_mask = orders["fill_event"].eq(1)

assert orders.loc[non_fill_mask, "first_fill_time_s"].isna().all()
assert orders.loc[non_fill_mask, "first_fill_quantity"].isna().all()
assert orders.loc[non_fill_mask, "first_fill_fraction"].isna().all()
assert orders.loc[fill_mask, "censoring_reason"].isna().all()

print("Missing values are consistent with eligibility and event status.")

Missing values are consistent with eligibility and event status.


In [18]:
chronology = (
    orders.assign(
        split_normalized=orders["data_split"].astype(str).str.lower(),
        trading_day_parsed=pd.to_datetime(orders["trading_day"])
    )
    .groupby("split_normalized")
    .agg(
        rows=("order_id", "size"),
        trading_days=("trading_day_parsed", "nunique"),
        first_day=("trading_day_parsed", "min"),
        last_day=("trading_day_parsed", "max")
    )
    .reindex(["train", "validation", "test"])
)

chronology

,rows,trading_days,first_day,last_day
split_normalized,,,,
train,7000,14,2022-09-06,2022-09-23
validation,1500,3,2022-09-26,2022-09-28
test,1500,3,2022-09-29,2022-10-03


In [19]:
assert (
    chronology.loc["train", "last_day"]
    < chronology.loc["validation", "first_day"]
)

assert (
    chronology.loc["validation", "last_day"]
    < chronology.loc["test", "first_day"]
)

print("Train, validation, and test periods are chronologically separated.")

Train, validation, and test periods are chronologically separated.


## Excel audit findings

- The `Orders` sheet contains 10,000 observations and 72 columns.
- Every `order_id` is present and unique.
- The supplied split contains 7,000 training, 1,500 validation, and
  1,500 test observations.
- The three splits use separate chronological trading periods.
- Eligibility and target availability are internally consistent.
- The positive fill rate increases from 5 to 30 to 60 seconds.
- Missing targets for ineligible observations must not be converted to zero.
- Post-observation outcome fields must not be used as model predictors.

In [21]:
from collections import Counter
import ijson

def open_json_binary(path):
    if path.name.endswith(".json.gz"):
        return gzip.open(path, mode="rb")
    return path.open(mode="rb")


order_count = 0
json_order_ids = set()
duplicated_json_order_ids = 0

history_length_counts = Counter()
invalid_sequence_orders = 0
nonnegative_time_offset_orders = 0

first_order_keys = None
first_snapshot_keys = None

with open_json_binary(JSON_PATH) as file:
    for order in ijson.items(file, "orders.item"):
        order_count += 1

        order_id = str(order["order_id"])

        if order_id in json_order_ids:
            duplicated_json_order_ids += 1

        json_order_ids.add(order_id)

        history = order.get("history", [])
        history_length_counts[len(history)] += 1

        sequence_steps = [
            snapshot.get("sequence_step")
            for snapshot in history
        ]

        if sequence_steps != list(range(1, len(history) + 1)):
            invalid_sequence_orders += 1

        if any(
            snapshot.get("milliseconds_before_submission", 0) >= 0
            for snapshot in history
        ):
            nonnegative_time_offset_orders += 1

        if first_order_keys is None:
            first_order_keys = sorted(order.keys())

            if history:
                first_snapshot_keys = sorted(history[0].keys())

print("Finished streaming JSON.")

Finished streaming JSON.


In [22]:
json_audit = pd.Series({
    "order_count": order_count,
    "unique_order_ids": len(json_order_ids),
    "duplicated_order_ids": duplicated_json_order_ids,
    "invalid_sequence_orders": invalid_sequence_orders,
    "nonnegative_time_offset_orders": nonnegative_time_offset_orders
})

json_audit

order_count                       10000
unique_order_ids                  10000
duplicated_order_ids                  0
invalid_sequence_orders               0
nonnegative_time_offset_orders        0
dtype: int64

In [23]:
history_count_table = pd.DataFrame(
    sorted(history_length_counts.items()),
    columns=["snapshots_per_order", "number_of_orders"]
)

history_count_table

,snapshots_per_order,number_of_orders
0,50,10000


In [24]:
excel_order_ids = set(
    orders["order_id"].astype(str)
)

missing_from_json = excel_order_ids - json_order_ids
extra_in_json = json_order_ids - excel_order_ids

join_audit = pd.Series({
    "excel_order_ids": len(excel_order_ids),
    "json_order_ids": len(json_order_ids),
    "missing_from_json": len(missing_from_json),
    "extra_in_json": len(extra_in_json)
})

join_audit

excel_order_ids      10000
json_order_ids       10000
missing_from_json        0
extra_in_json            0
dtype: int64

In [25]:
assert order_count == 10000
assert len(json_order_ids) == 10000
assert duplicated_json_order_ids == 0
assert history_length_counts == Counter({50: 10000})
assert invalid_sequence_orders == 0
assert nonnegative_time_offset_orders == 0
assert missing_from_json == set()
assert extra_in_json == set()

print("All JSON history checks passed.")

All JSON history checks passed.
